In [0]:
# Databricks notebook source
# Project: Azure Databricks Retail Supply Chain Lakehouse
# Notebook: 01-ingest-api-products
# Purpose: Ingest product data from the DummyJSON REST API.

import json
import requests

from datetime import datetime, timezone
from pyspark.sql import functions as F

API_URL = "https://dummyjson.com/products?limit=0"

LANDING_BASE_PATH = (
    "/Volumes/retail_dev/ops/"
    "landing_volume/api/dummyjson/products"
)

BRONZE_TABLE = "retail_dev.bronze.api_products"

In [0]:
response = requests.get(
    API_URL,
    timeout=30
)

response.raise_for_status()

payload = response.json()
products_count = len(payload["products"])

print(f"API status: {response.status_code}")
print(f"Products received: {products_count}")

In [0]:
ingestion_time = datetime.now(timezone.utc)

ingestion_date = ingestion_time.strftime("%Y-%m-%d")
file_timestamp = ingestion_time.strftime("%Y%m%dT%H%M%SZ")

raw_file_path = (
    f"{LANDING_BASE_PATH}/"
    f"ingestion_date={ingestion_date}/"
    f"products-{file_timestamp}.json"
)

dbutils.fs.put(
    raw_file_path,
    json.dumps(payload),
    overwrite=True
)

print(f"Raw API response saved to: {raw_file_path}")

In [0]:
raw_df = (
    spark.read
    .option("multiline", "true")
    .json(raw_file_path)
)

products_df = (
    raw_df
    .select(
        F.explode("products").alias("product")
    )
    .select("product.*")
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_system",
        F.lit("dummyjson_api")
    )
    .withColumn(
        "source_file",
        F.lit(raw_file_path)
    )
)

display(products_df)

In [0]:
(
    products_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(
    f"{products_df.count()} products loaded "
    f"into {BRONZE_TABLE}"
)

In [0]:
bronze_df = spark.table(BRONZE_TABLE)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Total rows: {bronze_df.count()}")

display(
    bronze_df.select(
        "id",
        "title",
        "category",
        "price",
        "stock",
        "availabilityStatus",
        "ingestion_timestamp"
    )
    .orderBy("id")
)